# Day 1 · Lab 1.3 · Under the hood
**Capgemini North America · AI Engineers & AI Quality Specialists**

In Lab 1.1 and 1.2 you built three architectures and an agent with memory on the n8n canvas.
Here is **the same thing with no canvas**: plain Python and the OpenAI library.

Run the cells **one at a time, top to bottom** (Shift + Enter). Wait for the trainer before each part.

| Part | Cells | Same as on the canvas |
|---|---|---|
| Setup | 1–2 | Install, paste the key, see `READY` |
| **A · Bare model** | 3–5 | Branch A: invents a price. Tokens and cost |
| **B · Grounded** | 6 | Branch B: the catalog pasted into the prompt |
| **C · Agent with tools** | 7–11 | Branch C: the model asks, your code runs the tool |
| **D · Memory** | 12 | Lab 1.2: Simple Memory = re-sending the history |
| **E · Break it** | 13 | Lab 1.2's break-it test, in code |

## Setup

In [1]:
# CELL 1 · Install the OpenAI library (about 10 seconds)
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 2 · Connect: paste the workshop key when asked (it stays hidden), then a ready check
import os, json, ast, operator
from getpass import getpass
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or getpass("Paste the workshop OpenAI key: "))
MODEL = "gpt-4.1-mini"

r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Reply with the single word READY"}])
print(r.choices[0].message.content)

READY


---
## Part A · Bare model (branch A)
One HTTP call: **messages in → text + token counts out.** Same question as Lab 1.1.

In [3]:
# CELL 3 · ONE call. Everything the model knows about this request is in `messages`
QUESTION = "What does it cost to certify 12 engineers in AI Foundations and 5 in DevOps Master, with the volume discount?"
PRICE = {"input": 0.40, "output": 1.60}   # USD per million tokens, gpt-4.1-mini list price

def call(messages, temperature=0.2):
    return client.chat.completions.create(model=MODEL, temperature=temperature, max_tokens=400, messages=messages)

def cost_usd(usage):
    return (usage.prompt_tokens * PRICE["input"] + usage.completion_tokens * PRICE["output"]) / 1_000_000

A_SYSTEM = "You are the Capgemini training desk assistant. Answer customer questions about courses and prices."
bare = [{"role": "system", "content": A_SYSTEM}, {"role": "user", "content": QUESTION}]   # exactly what branch A sends
r = call(bare)
print("ANSWER       :", r.choices[0].message.content)
print("FINISH REASON:", r.choices[0].finish_reason)      # 'stop' = finished, 'length' = hit max_tokens
print("TOKENS       :", r.usage.prompt_tokens, "in /", r.usage.completion_tokens, "out")
print(f"COST         : ${cost_usd(r.usage):.6f}")

ANSWER       : Here are the pricing details with volume discounts applied:

- AI Foundations certification:
  - Base price per engineer: $1,200
  - For 12 engineers, volume discount applies (typically 10% off for 10+ participants)
  - Discounted price per engineer: $1,200 - 10% = $1,080
  - Total for 12 engineers: 12 x $1,080 = $12,960

- DevOps Master certification:
  - Base price per engineer: $1,800
  - For 5 engineers, volume discount applies (typically 5% off for 5-9 participants)
  - Discounted price per engineer: $1,800 - 5% = $1,710
  - Total for 5 engineers: 5 x $1,710 = $8,550

Total cost for certifying 12 engineers in AI Foundations and 5 in DevOps Master with volume discounts:  
$12,960 + $8,550 = **$21,510**

Let me know if you need more details or assistance with registration!
FINISH REASON: stop
TOKENS       : 54 in / 228 out
COST         : $0.000386


In [4]:
# CELL 4 · Same question, three runs at a higher temperature. Watch the invented prices move
for i in range(3):
    print(f"run {i+1}:", call(bare, temperature=1.0).choices[0].message.content[:160], "...\n")
print("-> It has never seen our catalog. Each run predicts a different plausible number.")

run 1: The AI Foundations course costs €600 per engineer, and the DevOps Master course costs €1,100 per engineer.

With volume discounts for 10 or more engineers:
- AI ...

run 2: The prices per certification course are:

- AI Foundations: €500 per engineer
- DevOps Master: €800 per engineer

For volume discounts:

- 10 to 19 certificatio ...

run 3: For certifying engineers, here are the prices per certification:

- AI Foundations: €300 per engineer
- DevOps Master: €450 per engineer

The volume discount ap ...

-> It has never seen our catalog. Each run predicts a different plausible number.


In [5]:
# CELL 5 · No memory: a follow-up in a NEW call
call([{"role": "user", "content": "My name is Priya."}])
r = call([{"role": "user", "content": "What is my name?"}])
print("ANSWER:", r.choices[0].message.content)
print("-> Nothing from the first call was sent in the second. The model is stateless.")

ANSWER: I don't have access to your personal information, so I don't know your name. How can I assist you today?
-> Nothing from the first call was sent in the second. The model is stateless.


---
## Part B · Grounded (branch B)
The **same call**, with the catalog pasted into the prompt. That's the last step of every RAG system: retrieval only decides *which* text goes in this block.

In [6]:
# CELL 6 · B · Put the facts in the prompt
CATALOG_TEXT = """APPROVED COURSE CATALOG (source: catalog.csv, refreshed 2026-09-01)
AIF-101 | AI Foundations        | 4 days | USD 1200 per seat
DOM-300 | DevOps Master         | 5 days | USD 1850 per seat
ITL-4F  | ITIL 4 Foundation     | 3 days | USD 950 per seat
AIQ-200 | AI Quality Specialist | 4 days | USD 1450 per seat

POLICY: Volume discount of 10 percent applies from 10 seats of the same course."""

grounded = [{"role": "user", "content":
    "Answer ONLY from the retrieved context below. If the context does not contain the answer, say so.\n\n"
    f"<retrieved_context>\n{CATALOG_TEXT}\n</retrieved_context>\n\nQUESTION: {QUESTION}"}]
r = call(grounded)
print("ANSWER:", r.choices[0].message.content)
print("\nTOKENS:", r.usage.prompt_tokens, "in  (cell 3 was much smaller: the context block costs tokens on every call)")
print("-> Right prices now. But the arithmetic is still next-token prediction, not calculation.")

ANSWER: The cost to certify 12 engineers in AI Foundations with the volume discount is:

- AI Foundations: USD 1200 per seat
- 12 seats qualify for 10% discount
- Discounted price per seat = 1200 - (10% of 1200) = 1200 - 120 = USD 1080
- Total for 12 seats = 12 × 1080 = USD 12,960

The cost to certify 5 engineers in DevOps Master (no discount since less than 10 seats):

- DevOps Master: USD 1850 per seat
- Total for 5 seats = 5 × 1850 = USD 9,250

Total cost = USD 12,960 + USD 9,250 = USD 22,210

TOKENS: 188 in  (cell 3 was much smaller: the context block costs tokens on every call)
-> Right prices now. But the arithmetic is still next-token prediction, not calculation.


---
## Part C · Agent with tools (branch C)
The model **asks** for a tool. **Your code** runs it and sends the result back. Repeat until it answers.

In [7]:
# CELL 7 · The tools are ordinary Python functions (same catalog and calculator as the n8n tools)
CATALOG = [
    {"code": "AIF-101", "name": "AI Foundations",        "days": 4, "price_usd": 1200},
    {"code": "DOM-300", "name": "DevOps Master",         "days": 5, "price_usd": 1850},
    {"code": "ITL-4F",  "name": "ITIL 4 Foundation",     "days": 3, "price_usd": 950},
    {"code": "AIQ-200", "name": "AI Quality Specialist", "days": 4, "price_usd": 1450},
]
POLICY = "POLICY: 10 percent volume discount when 10 or more seats of the SAME course are booked in one order."

def course_catalog_lookup(query: str) -> str:
    q = query.lower()
    hits = [c for c in CATALOG if c["name"].lower() in q or c["code"].lower() in q] or CATALOG
    return "\n".join(f'{c["code"]} | {c["name"]} | {c["days"]} days | USD {c["price_usd"]} per seat' for c in hits) + "\n" + POLICY

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}

def calculator(expression: str) -> str:
    """Safe arithmetic only. Never eval() text a model wrote."""
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp):   return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp): return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("only + - * / ** and numbers are allowed")
    return str(round(ev(ast.parse(expression, mode="eval").body), 2))

FUNCTIONS = {"course_catalog_lookup": course_catalog_lookup, "calculator": calculator}
print(course_catalog_lookup("DevOps Master"))
print("calculator('12*1200*0.9') =", calculator("12*1200*0.9"))

DOM-300 | DevOps Master | 5 days | USD 1850 per seat
POLICY: 10 percent volume discount when 10 or more seats of the SAME course are booked in one order.
calculator('12*1200*0.9') = 12960.0


In [8]:
# CELL 8 · ...described to the model as JSON. The description is ALL the model ever sees
TOOLS = [
    {"type": "function", "function": {
        "name": "course_catalog_lookup",
        "description": "Look up approved course codes, durations and per-seat prices. Pass the course name or code as the query.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string", "description": "Course name or code"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression, e.g. '12*1200*0.9 + 5*1850'.",
        "parameters": {"type": "object",
                       "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
]
SYSTEM = ("You are the Capgemini training desk assistant. Never quote a price from memory: always call "
          "course_catalog_lookup first. Never do arithmetic in your head: always call calculator, including for "
          "the final grand total. Show each course's per-seat price, seat count and any discount, then end with "
          "one line: 'Grand total: USD <amount>'. If a course is not in the catalog, say so.")
print(len(TOOLS), "tools described. Same system message as the n8n agent.")

2 tools described. Same system message as the n8n agent.


In [9]:
# CELL 9 · UNDER THE HOOD: what the model sends back when it wants a tool
first = client.chat.completions.create(model=MODEL, temperature=0, tools=TOOLS, messages=[
    {"role": "system", "content": SYSTEM},
    {"role": "user",   "content": "What does DevOps Master cost per seat?"}])
msg = first.choices[0].message
print("TEXT ANSWER :", msg.content)
for c in msg.tool_calls or []:
    print("TOOL REQUEST:", json.dumps({"tool": c.function.name, "arguments": json.loads(c.function.arguments)}))
print("-> No answer, just JSON asking YOUR code to run a function. The model can't run anything itself.")

TEXT ANSWER : None
TOOL REQUEST: {"tool": "course_catalog_lookup", "arguments": {"query": "DevOps Master"}}
-> No answer, just JSON asking YOUR code to run a function. The model can't run anything itself.


In [10]:
# CELL 10 · The agent loop (what n8n's AI Agent node does): call -> tool requested? -> WE run it -> send back -> repeat
def run_agent(question, history=None, max_steps=6):
    messages = [{"role": "system", "content": SYSTEM}] + (history or []) + [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(model=MODEL, temperature=0.2,
                                             messages=messages, tools=TOOLS).choices[0].message
        if not msg.tool_calls:                                  # no tool requested -> this is the answer
            return msg.content, messages + [{"role": "assistant", "content": msg.content}]
        messages.append(msg.model_dump(exclude_none=True))      # keep the model's request
        for call_ in msg.tool_calls:
            args = json.loads(call_.function.arguments)
            result = FUNCTIONS[call_.function.name](**args)     # YOUR code runs the tool, not the model
            print(f"  step {step}: model asked for {call_.function.name}({args}) -> {result.splitlines()[0]}")
            messages.append({"role": "tool", "tool_call_id": call_.id, "content": result})
    return "Stopped: too many steps (always cap the loop).", messages

print("run_agent() is ready")

run_agent() is ready


In [11]:
# CELL 11 · Branch C, in code. Expected: Grand total USD 22,210 (10% off AI Foundations only)
answer, convo = run_agent(QUESTION)
print("\nANSWER:\n" + answer)

  step 1: model asked for course_catalog_lookup({'query': 'AI Foundations'}) -> AIF-101 | AI Foundations | 4 days | USD 1200 per seat
  step 1: model asked for course_catalog_lookup({'query': 'DevOps Master'}) -> DOM-300 | DevOps Master | 5 days | USD 1850 per seat
  step 2: model asked for calculator({'expression': '12 * 1200 * 0.9 + 5 * 1850'}) -> 22210.0

ANSWER:
The AI Foundations course costs USD 1200 per seat, and with 12 seats booked, a 10% volume discount applies. The DevOps Master course costs USD 1850 per seat, but with only 5 seats, no volume discount applies.

Calculation:
- AI Foundations: 12 seats * USD 1200 * 0.9 (10% discount)
- DevOps Master: 5 seats * USD 1850

Grand total: USD 22210.0


---
## Part D · Memory (Lab 1.2)
Simple Memory in n8n = **your code re-sending the conversation** on every call.

In [12]:
# CELL 12 · The same follow-up, without and with the history
print("WITHOUT history:")
print(run_agent("And if it were 4 DevOps Master seats instead of 5?")[0], "\n")
print("WITH history:")
answer2, _ = run_agent("And if it were 4 DevOps Master seats instead of 5?", history=convo[1:])
print(answer2)
print("\n-> 'It' only means something because we sent the earlier turns again. You pay for those tokens every call.")

WITHOUT history:
  step 1: model asked for course_catalog_lookup({'query': 'DevOps Master'}) -> DOM-300 | DevOps Master | 5 days | USD 1850 per seat
  step 2: model asked for calculator({'expression': '4*1850'}) -> 7400
For 4 DevOps Master seats, the price per seat is USD 1850. Since the volume discount applies only for 10 or more seats, no discount is applied here.

Grand total: USD 7400 

WITH history:
  step 1: model asked for calculator({'expression': '12 * 1200 * 0.9 + 4 * 1850'}) -> 20360.0
With 4 seats for DevOps Master instead of 5, the calculation is:
- AI Foundations: 12 seats * USD 1200 * 0.9 (10% discount)
- DevOps Master: 4 seats * USD 1850 (no discount)

Grand total: USD 20360.0

-> 'It' only means something because we sent the earlier turns again. You pay for those tokens every call.


---
## Part E · Break it

In [13]:
# CELL 13 · Change the question and run again. Try the five break-it inputs from Lab 1.2
print(run_agent("What does the Quantum Computing Masterclass cost?")[0])

  step 1: model asked for course_catalog_lookup({'query': 'Quantum Computing Masterclass'}) -> AIF-101 | AI Foundations | 4 days | USD 1200 per seat
The Quantum Computing Masterclass is not listed in the course catalog. If you have another course in mind or need information on any of the listed courses, please let me know!
